In [4]:
import cv2

In [5]:
   
face_cascade=cv2.CascadeClassifier( r"haar/haarcascade_frontalface_default.xml")

In [6]:
cap=cv2.VideoCapture("fariz.jpg")

In [7]:
while True:
    ret, frame = cap.read()  
    if not ret:
        break  

    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        face_img = frame[y:y+h, x:x+w] 
    

 
    cv2.imshow("Face Detection", frame)

   
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

# face_i = cv2.resize(face_img, (160, 160))  # Resize for FaceNet
# face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)  # Convert to RGB
# face_normalized = face_rgb / 255.0  # Normalize pixels



In [8]:
import cv2
import torch
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from PIL import Image
import os
import torch.nn.functional as F

# -----------------------------
# 1. Load Haar Cascade for face detection
# -----------------------------
face_cascade = cv2.CascadeClassifier(r"haar/haarcascade_frontalface_default.xml")


# -----------------------------
# 2. Load FaceNet Model (PyTorch)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = InceptionResnetV1(pretrained='vggface2').eval().to(device)
print("FaceNet model loaded on", device)

# -----------------------------
# 3. Preprocessing Pipeline
# -----------------------------
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

# -----------------------------
# 4. Load Known Faces and Generate Embeddings
# -----------------------------
KNOWN_FACES_DIR = "known_faces"
known_embeddings = []
known_names = []

for filename in os.listdir(KNOWN_FACES_DIR):
    if filename.endswith(".jpg") or filename.endswith(".png"):
        name = os.path.splitext(filename)[0]
        img_path = os.path.join(KNOWN_FACES_DIR, filename)
        
        # Load PIL image
        img = Image.open(img_path).resize((160,160))
        img_tensor = preprocess(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            emb = model(img_tensor)
        
        known_embeddings.append(emb)
        known_names.append(name)

print("Known faces loaded:", known_names)

# -----------------------------
# 5. Function to Recognize Face
# -----------------------------
def recognize_face(face_img):
    face_pil = Image.fromarray(cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)).resize((160,160))
    face_tensor = preprocess(face_pil).unsqueeze(0).to(device)
    
    with torch.no_grad():
        emb = model(face_tensor)
    
    name = "Unknown"
    min_dist = 1e6
    
    for i, db_emb in enumerate(known_embeddings):
        dist = F.pairwise_distance(emb, db_emb).item()
        if dist < 0.6 and dist < min_dist:
            name = known_names[i]
            min_dist = dist
    
    return name

# -----------------------------
# 6. Start Camera and Detect + Recognize Faces
# -----------------------------
cap = cv2.VideoCapture(0) 
scaleFactor = 1.1       # Smaller increments detect smaller faces
minNeighbors = 6        # Higher value reduces false positives
minSize = (50, 50)      # Ignore very small regions (likely background edges)
maxSize = (300, 300)    
while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Optional: Apply slight blur to reduce background edges detection
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=scaleFactor,
        minNeighbors=minNeighbors,
        minSize=minSize,
        maxSize=maxSize
    )

    # Draw rectangles around faces
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Display live frame
    cv2.imshow("Face Detection", frame)

    # Press 'q' to quit
    if cv2.waitKey(0) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

FaceNet model loaded on cpu
Known faces loaded: []


In [ ]:
import cv2
import torch
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from PIL import Image
import os
import torch.nn.functional as F
from ultralytics import YOLO

# -----------------------------
# 1. Load YOLO Face Detector
# -----------------------------
yolo_model = YOLO("model.pt")

# -----------------------------
# 2. Load FaceNet Model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
facenet_model = InceptionResnetV1(
    pretrained='vggface2'
).eval().to(device)

print("FaceNet loaded on:", device)

# -----------------------------
# 3. Preprocessing
# -----------------------------
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,
    0.5], [0.5,0.5,0.5])
])

# -----------------------------
# 4. Load Known Faces
# -----------------------------
KNOWN_FACES_DIR = "known_faces"
known_embeddings = []
known_names = []

for file in os.listdir(KNOWN_FACES_DIR):
    if file.endswith(".jpg") or file.endswith(".png"):
        name = os.path.splitext(file)[0]
        img_path = os.path.join(KNOWN_FACES_DIR, file)

        img = Image.open(img_path).resize((160,160))
        img_tensor = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():
            emb = facenet_model(img_tensor)

        known_embeddings.append(emb)
        known_names.append(name)

print("Known faces:", known_names)

# -----------------------------
# 5. Face Recognition Function
# -----------------------------
def recognize_face(face_img):
    face_pil = Image.fromarray(
        cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    ).resize((160,160))

    face_tensor = preprocess(face_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = facenet_model(face_tensor)

    best_dist = float("inf")
    best_name= "Unknown"

    for i, db_emb in enumerate(known_embeddings):
        dist = F.pairwise_distance(emb, db_emb).item()

        if dist < best_dist:
            best_dist = dist
            best_name = known_names[i]

    # 🔑 THRESHOLD (IMPORTANT)
    if best_dist < 0.9:
        return best_name
    else:
        return "Unknown"


# -----------------------------
# 6. Camera Loop (FIXED)
# -----------------------------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = yolo_model(frame, conf=0.5)

    for r in results:
        for box in r.boxes:
            


            x1, y1, x2, y2 = map(int, box.xyxy[0])

            face = frame[y1:y2, x1:x2]
            if face.size == 0:
                continue

            name = recognize_face(face)

            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(
                frame, name, (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (0,255,0), 2
            )

    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


ModuleNotFoundError: No module named 'facenet_pytorch'

In [10]:
cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import torch
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from PIL import Image
import os
import torch.nn.functional as F
from ultralytics import YOLO

# -----------------------------
# 1. YOLO Face Detector
# -----------------------------
yolo_model = YOLO("model.pt")

# -----------------------------
# 2. FaceNet Model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
facenet_model = InceptionResnetV1(
    pretrained='vggface2'
).eval().to(device)

# -----------------------------
# 3. Preprocessing
# -----------------------------
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

# -----------------------------
# 4. Load Known Faces (SUBFOLDERS)
# -----------------------------
KNOWN_FACES_DIR = "known_faces"
known_embeddings = []
known_names = []

for person_name in os.listdir(KNOWN_FACES_DIR):
    person_dir = os.path.join(KNOWN_FACES_DIR, person_name)
    if not os.path.isdir(person_dir):
        continue

    for img_file in os.listdir(person_dir):
        if img_file.endswith(".jpg") or img_file.endswith(".png"):
            img_path = os.path.join(person_dir, img_file)

            img = Image.open(img_path).resize((160,160))
            img_tensor = preprocess(img).unsqueeze(0).to(device)

            with torch.no_grad():
                emb = facenet_model(img_tensor)

            known_embeddings.append(emb)
            known_names.append(person_name)

print("Loaded identities:", set(known_names))
print("Total embeddings:", len(known_embeddings))

# -----------------------------
# 5. Recognition Function
# -----------------------------
def recognize_face(face_img):
    face_pil = Image.fromarray(
        cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    ).resize((160,160))

    face_tensor = preprocess(face_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = facenet_model(face_tensor)

    best_dist = float("inf")
    best_name = "Unknown"

    for i, db_emb in enumerate(known_embeddings):
        dist = F.pairwise_distance(emb, db_emb).item()
        if dist < best_dist:
            best_dist = dist
            best_name = known_names[i]

    # THRESHOLD
    if best_dist < 0.5:
        return best_name
    else:
        return "Unknown"

# -----------------------------
# 6. Camera Loop
# -----------------------------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = yolo_model(frame, conf=0.5)

    h, w, _ = frame.shape
    pad = 20

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Add padding
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(w, x2 + pad)
            y2 = min(h, y2 + pad)

            face = frame[y1:y2, x1:x2]
            if face.size == 0:
                continue

            name = recognize_face(face)

            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, name, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0,255,0), 2)
            print("Faces detected:", len(r.boxes))

    cv2.imshow("Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Loaded identities: {'andriana', 'harish', 'fariz'}
Total embeddings: 7


In [ ]:
import cv2
import torch
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from PIL import Image
import os
import torch.nn.functional as F
from ultralytics import YOLO

# -----------------------------
# 1. YOLO Face Detector
# -----------------------------
yolo_model = YOLO("model.pt")  # your YOLO face detection model

# -----------------------------
# 2. FaceNet Model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
facenet_model = InceptionResnetV1(
    pretrained='vggface2'
).eval().to(device)

# -----------------------------
# 3. Preprocessing
# -----------------------------
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

# -----------------------------
# 4. Load Known Faces (SUBFOLDERS)
# -----------------------------
KNOWN_FACES_DIR = "known_faces"
known_embeddings = []
known_names = []

for person_name in os.listdir(KNOWN_FACES_DIR):
    person_dir = os.path.join(KNOWN_FACES_DIR, person_name)
    if not os.path.isdir(person_dir):
        continue

    for img_file in os.listdir(person_dir):
        if img_file.endswith(".jpg") or img_file.endswith(".png"):
            img_path = os.path.join(person_dir, img_file)

            img = Image.open(img_path).convert("RGB").resize((160,160))
            img_tensor = preprocess(img).unsqueeze(0).to(device)

            with torch.no_grad():
                emb = facenet_model(img_tensor)

            known_embeddings.append(emb)
            known_names.append(person_name)
            print(f"Loaded embedding for {person_name}:")
            print(emb)  # <-- Print known embedding vector

print("\nAll known embeddings loaded.")
print("Known identities:", set(known_names))
print("Total embeddings:", len(known_embeddings))

# -----------------------------
# 5. Recognition Function (with Distance Printing)
# -----------------------------
def recognize_face(face_img):
    face_pil = Image.fromarray(
        cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
    ).resize((160,160))

    face_tensor = preprocess(face_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = facenet_model(face_tensor)

    print("\nDetected face embedding:")
    print(emb)  # <-- Print embedding of detected face

    best_dist = float("inf")
    best_name = "Unknown"

    # Compare with all known embeddings
    for i, db_emb in enumerate(known_embeddings):
        dist = F.pairwise_distance(emb, db_emb).item()
        print(f"Distance to {known_names[i]}: {dist:.4f}")  # <-- Print distance
        if dist < best_dist:
            best_dist = dist
            best_name = known_names[i]

    # Apply threshold
    if best_dist < 1:
        return best_name, best_dist
    else:
        return "Unknown", best_dist

# -----------------------------
# 6. Camera Loop
# -----------------------------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = yolo_model(frame, conf=0.5)
    h, w, _ = frame.shape
    pad = 20

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Add padding
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(w, x2 + pad)
            y2 = min(h, y2 + pad)

            face = frame[y1:y2, x1:x2]
            if face.size == 0:
                continue

            name, distance = recognize_face(face)

            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, f"{name} ({distance:.2f})", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0,255,0), 2)
            print(f"Faces detected: {len(r.boxes)}, Recognized as: {name}, Distance: {distance:.4f}")

    cv2.imshow("Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Loaded embedding for andriana:
tensor([[ 0.0343, -0.0419,  0.0027, -0.0731,  0.0365,  0.0501,  0.0596, -0.0585,  0.0095, -0.0451,  0.0035,  0.0011, -0.0238, -0.0309, -0.0016, -0.0527, -0.0173, -0.0427,  0.0074,  0.0089, -0.0365, -0.0064,  0.0827, -0.0021, -0.0186,  0.0043, -0.0237, -0.0635, -0.0180, -0.0591,  0.0313, -0.0039, -0.0541, -0.0367,
          0.0497,  0.0505,  0.0254, -0.0355, -0.0121,  0.0353,  0.0759, -0.0442, -0.0103, -0.0216,  0.0051, -0.0253, -0.0005,  0.1211, -0.0504, -0.0074,  0.0520,  0.0099, -0.0481,  0.0420,  0.0583,  0.0559,  0.0231, -0.0072,  0.0399, -0.0148,  0.0015,  0.0561,  0.0171,  0.0102,  0.0292, -0.0510,  0.0113, -0.0544,
         -0.0116, -0.0083, -0.0103,  0.0618, -0.0526, -0.0492, -0.0068, -0.0621,  0.0249, -0.1105, -0.0169,  0.0105, -0.0297,  0.0403, -0.0803,  0.0184,  0.0295, -0.0123,  0.0365, -0.0728,  0.0141,  0.0119,  0.0929, -0.0269,  0.0747, -0.0760, -0.0103, -0.0064, -0.0550,  0.0689,  0.0036, -0.0069,  0.0925, -0.0460,
         -0.0241,  0.023

In [ ]:
cap.release()
cv2.destroyAllWindows()

In [1]:
import cv2
import torch
import os
from facenet_pytorch import MTCNN, InceptionResnetV1
import torch.nn.functional as F

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# MTCNN
# -----------------------------
mtcnn = MTCNN(
    image_size=160,
    margin=20,
    keep_all=True,
    device=device
)

# -----------------------------
# FaceNet
# -----------------------------
facenet = InceptionResnetV1(
    pretrained="vggface2"
).eval().to(device)

# -----------------------------
# Load Known Faces
# -----------------------------
KNOWN_FACES_DIR = "known_faces"
known_embeddings = []
known_names = []

for name in os.listdir(KNOWN_FACES_DIR):
    person_dir = os.path.join(KNOWN_FACES_DIR, name)
    if not os.path.isdir(person_dir):
        continue

    embeddings = []

    for img_name in os.listdir(person_dir):
        img_path = os.path.join(person_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        faces = mtcnn(rgb)

        if faces is None:
            continue

        # faces shape: [N, 3, 160, 160]
        with torch.no_grad():
            embs = facenet(faces.to(device))
            embeddings.append(embs)

    if embeddings:
        avg_emb = torch.mean(torch.cat(embeddings), dim=0, keepdim=True)
        known_embeddings.append(avg_emb)
        known_names.append(name)

print("Loaded identities:", known_names)

# -----------------------------
# Recognition Function
# -----------------------------
def recognize_face(face_emb, threshold=0.8):
    min_dist = float("inf")
    identity = "Unknown"

    for i, known_emb in enumerate(known_embeddings):
        dist = F.pairwise_distance(face_emb, known_emb).item()
        print(f"Distance to {known_names[i]}: {dist:.4f}")

        if dist < min_dist:
            min_dist = dist
            identity = known_names[i]

    if min_dist < threshold:
        return identity, min_dist
    else:
        return "Unknown", min_dist

# -----------------------------
# Webcam Loop
# -----------------------------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    boxes, _ = mtcnn.detect(rgb)

    if boxes is not None:
        faces = mtcnn(rgb)

        with torch.no_grad():
            embs = facenet(faces.to(device))

        for box, emb in zip(boxes, embs):
            x1, y1, x2, y2 = map(int, box)

            name, dist = recognize_face(emb.unsqueeze(0))

            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(
                frame,
                f"{name} ({dist:.2f})",
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0,255,0),
                2
            )

    cv2.imshow("Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Loaded identities: ['andriana', 'chandru', 'fariz', 'harish', 'jeeva', 'yashwanth']
Distance to andriana: 1.3582
Distance to chandru: 0.9961
Distance to fariz: 0.7092
Distance to harish: 0.9828
Distance to jeeva: 1.1175
Distance to yashwanth: 1.1437
Distance to andriana: 1.3779
Distance to chandru: 0.9778
Distance to fariz: 0.7025
Distance to harish: 0.9885
Distance to jeeva: 1.0816
Distance to yashwanth: 1.1228
Distance to andriana: 1.3674
Distance to chandru: 0.9750
Distance to fariz: 0.7135
Distance to harish: 0.9993
Distance to jeeva: 1.0930
Distance to yashwanth: 1.1016
Distance to andriana: 1.3671
Distance to chandru: 0.9850
Distance to fariz: 0.7021
Distance to harish: 0.9743
Distance to jeeva: 1.0541
Distance to yashwanth: 1.1242
Distance to andriana: 1.3845
Distance to chandru: 0.9949
Distance to fariz: 0.6959
Distance to harish: 1.0098
Distance to jeeva: 1.1072
Distance to yashwanth: 1.1343
Distance to andriana: 1.3584
Distance to chandru: 1.0027
Distance to fariz: 0.7037
Dis

In [9]:
MODEL_PATH = "face_recognition_model.pth"

torch.save({
    "known_embeddings": known_embeddings,
    "known_names": known_names
}, MODEL_PATH)

print("✅ Face recognition model saved!")


✅ Face recognition model saved!


In [7]:
import cv2
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import torch.nn.functional as F

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Load MTCNN & FaceNet
# -----------------------------
mtcnn = MTCNN(image_size=160, margin=20, keep_all=True, device=device)

facenet = InceptionResnetV1(
    pretrained="vggface2"
).eval().to(device)

# -----------------------------
# Load Saved Model
# -----------------------------
checkpoint = torch.load("face_recognition_model.pth", map_location=device)

known_embeddings = checkpoint["known_embeddings"]
known_names = checkpoint["known_names"]

print("✅ Loaded identities:", known_names)

# -----------------------------
# Recognition Function
# -----------------------------
def recognize_face(face_emb, threshold=0.8):
    min_dist = float("inf")
    identity = "Unknown"

    for i, known_emb in enumerate(known_embeddings):
        dist = F.pairwise_distance(face_emb, known_emb).item()
        if dist < min_dist:
            min_dist = dist
            identity = known_names[i]

    if min_dist < threshold:
        return identity, min_dist
    else:
        return "Unknown", min_dist

# -----------------------------
# Webcam
# -----------------------------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    boxes, _ = mtcnn.detect(rgb)

    if boxes is not None:
        faces = mtcnn(rgb)

        with torch.no_grad():
            embs = facenet(faces.to(device))

        for box, emb in zip(boxes, embs):
            x1, y1, x2, y2 = map(int, box)
            name, dist = recognize_face(emb.unsqueeze(0))

            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(
                frame,
                f"{name} ({dist:.2f})",
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0,255,0),
                2
            )

    cv2.imshow("Saved Face Recognition Model", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


✅ Loaded identities: ['andriana', 'chandru', 'fariz', 'harish', 'jeeva', 'yashwanth']


In [6]:
import cv2
import torch
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from PIL import Image
import os
import torch.nn.functional as F
from ultralytics import YOLO
import torchvision.transforms.functional as TF

# -----------------------------
# 1. YOLO Face Detector
# -----------------------------
yolo_model = YOLO("model.pt")

# -----------------------------
# 2. FaceNet Model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
facenet_model = InceptionResnetV1(
    pretrained='vggface2'
).eval().to(device)

# -----------------------------
# 3. Preprocessing
# -----------------------------
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

# -----------------------------
# 4. Load Known Faces (SUBFOLDERS)
# -----------------------------
KNOWN_FACES_DIR = "known_faces"
known_embeddings = []
known_names = []

for person_name in os.listdir(KNOWN_FACES_DIR):
    person_dir = os.path.join(KNOWN_FACES_DIR, person_name)
    if not os.path.isdir(person_dir):
        continue

    for img_file in os.listdir(person_dir):
        if img_file.endswith(".jpg") or img_file.endswith(".png"):
            img_path = os.path.join(person_dir, img_file)

            img = Image.open(img_path).resize((160,160))
            img_tensor = preprocess(img).unsqueeze(0).to(device)

            with torch.no_grad():
                emb = facenet_model(img_tensor)

            known_embeddings.append(emb)
            known_names.append(person_name)

print("Loaded identities:", set(known_names))
print("Total embeddings:", len(known_embeddings))
def align_face_tensor(face_bgr, device):
    """
    face_bgr : numpy array (H,W,3) from YOLO crop
    returns  : aligned face tensor (1,3,160,160)
    """

    # BGR → RGB
    face_rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)

    # Convert to tensor [3,H,W]
    face_tensor = torch.from_numpy(face_rgb).permute(2,0,1).float()

    h, w = face_tensor.shape[1:]

    # -------- square crop (center) --------
    size = min(h, w)
    y1 = (h - size) // 2
    x1 = (w - size) // 2

    face_tensor = face_tensor[:, y1:y1+size, x1:x1+size]

    # -------- resize to FaceNet input --------
    face_tensor = TF.resize(face_tensor, [160,160])

    # -------- normalize --------
    face_tensor = face_tensor / 255.0
    face_tensor = (face_tensor - 0.5) / 0.5

    return face_tensor.unsqueeze(0).to(device)
# -----------------------------
# 5. Recognition Function
# -----------------------------
def recognize_face(face_img):
    face_tensor = align_face_tensor(face_img, device)

    with torch.no_grad():
        emb = facenet_model(face_tensor)

    best_dist = float("inf")
    best_name = "Unknown"

    for i, db_emb in enumerate(known_embeddings):
        dist = F.pairwise_distance(emb, db_emb).item()
        if dist < best_dist:
            best_dist = dist
            best_name = known_names[i]

    print(f"Min distance: {best_dist:.3f}")

    if best_dist < 0.6:
        return best_name
    else:
        return "Unknown"


# -----------------------------
# 6. Camera Loop
# -----------------------------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = yolo_model(frame, conf=0.5)

    h, w, _ = frame.shape
    pad = 20

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Add padding
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(w, x2 + pad)
            y2 = min(h, y2 + pad)

            face = frame[y1:y2, x1:x2]
            if face.size == 0:
                continue

            name = recognize_face(face)

            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, name, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0,255,0), 2)
            print("Faces detected:", len(r.boxes))

    cv2.imshow("Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Loaded identities: {'andriana', 'jeeva', 'chandru', 'yashwanth', 'harish', 'fariz', 'fawad'}
Total embeddings: 30

0: 480x640 1 face, 171.4ms
Speed: 12.5ms preprocess, 171.4ms inference, 5.2ms postprocess per image at shape (1, 3, 480, 640)
Min distance: 0.615
Faces detected: 1

0: 480x640 1 face, 176.0ms
Speed: 7.5ms preprocess, 176.0ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
Min distance: 0.587
Faces detected: 1

0: 480x640 2 faces, 130.6ms
Speed: 3.3ms preprocess, 130.6ms inference, 6.4ms postprocess per image at shape (1, 3, 480, 640)
Min distance: 1.240
Faces detected: 2
Min distance: 1.082
Faces detected: 2

0: 480x640 2 faces, 120.5ms
Speed: 2.2ms preprocess, 120.5ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)
Min distance: 1.245
Faces detected: 2
Min distance: 1.026
Faces detected: 2

0: 480x640 2 faces, 121.2ms
Speed: 3.0ms preprocess, 121.2ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)
Min distance: 1.244
Face